Azure provides several load-balancing services, each operating at a different layer of the network stack:

| Service | Layer | Scope | Use case |
|---|---|---|---|
| **Azure Load Balancer** | L4 (TCP/UDP) | Regional | High-throughput, low-latency distribution across VM instances |
| **Application Gateway** | L7 (HTTP/HTTPS) | Regional | Web application delivery with SSL termination, WAF, URL routing |
| **Azure Front Door** | L7 (HTTP/HTTPS) | Global | Global HTTP load balancing, CDN, WAF across regions |
| **Azure Traffic Manager** | DNS | Global | DNS-based routing across regions or endpoints |

This topic covers the two most commonly tested services: **Azure Load Balancer** and **Application Gateway**.

## Azure Load Balancer

**Azure Load Balancer** is a Layer 4 (TCP/UDP) load balancer that distributes inbound traffic across healthy backend instances using a 5-tuple hash. It operates at network speed with no TLS termination, no HTTP awareness, and no session persistence based on application cookies.

The Azure equivalent of AWS Network Load Balancer (NLB).

### SKUs

| Feature | Basic | Standard |
|---|---|---|
| Backend pool size | Up to 300 | Up to 1,000 |
| Health probe types | HTTP, TCP | HTTP, HTTPS, TCP |
| Availability Zones | No | Yes (zone-redundant and zonal) |
| SLA | None | 99.99% |
| Outbound rules | No | Yes |
| Multi-frontend | No | Yes |
| HA ports | No | Yes |
| Cross-region load balancing | No | Yes (via cross-region LB) |

> Always use **Standard SKU** for new deployments. Basic is being retired.

### Core components

```
Frontend IP config  →  Load-balancing rules  →  Backend pool
                           ↕
                       Health probes
```

| Component | Description |
|---|---|
| **Frontend IP** | Public or private IP that clients connect to |
| **Backend pool** | Set of VMs or VMSS instances that receive traffic |
| **Health probe** | Periodic check (HTTP/HTTPS/TCP) — unhealthy instances are removed from rotation |
| **Load-balancing rule** | Maps frontend IP:port → backend pool:port using the distribution algorithm |
| **Inbound NAT rule** | Maps frontend IP:port directly to a specific backend VM (e.g. for SSH/RDP) |
| **Outbound rule** | Configures SNAT for backend pool instances to reach the internet |

### Distribution algorithm

Default: **5-tuple hash** — source IP, source port, destination IP, destination port, protocol. Same 5-tuple always maps to the same backend.

Session persistence modes ("stickiness"):

| Mode | Hash | Effect |
|---|---|---|
| **None** (default) | 5-tuple | Each new connection may go to any backend |
| **Client IP** | 2-tuple (src IP + dst IP) | All connections from same client IP go to same backend |
| **Client IP and protocol** | 3-tuple | Same client IP + protocol → same backend |

### Public vs Internal Load Balancer

| Type | Frontend IP | Use case |
|---|---|---|
| **Public** | Public IP | Internet-facing — distribute inbound internet traffic to VMs |
| **Internal (ILB)** | Private IP (VNet) | Internal — distribute traffic between tiers (e.g. web → app) |

### HA Ports

Setting the port to **0** and the protocol to **All** on an internal Standard Load Balancer creates an **HA Ports** rule — it forwards all TCP/UDP traffic on all ports to the backend pool simultaneously. This is used for **Network Virtual Appliances (NVAs)** like firewalls in a load-balanced active-active configuration.

### Cross-region Load Balancer

A **Cross-region Load Balancer** sits in front of regional Standard Load Balancers and distributes traffic across Azure regions. It provides a single global IP and automatically routes traffic to the closest healthy region.

## Application Gateway

**Azure Application Gateway** is a Layer 7 (HTTP/HTTPS) load balancer and application delivery controller. Unlike Azure Load Balancer, it understands HTTP — it can route based on URL path, hostname, or headers, terminate SSL/TLS, rewrite HTTP headers, and optionally apply a Web Application Firewall.

The Azure equivalent of AWS Application Load Balancer (ALB) + AWS WAF.

### SKUs

| SKU | Autoscaling | WAF | Zone support |
|---|---|---|---|
| **Standard v2** | Yes (0–125 capacity units) | No | Yes |
| **WAF v2** | Yes | Yes | Yes |

> v1 SKUs are retired — always use v2.

### Core components

```
Listener  →  Request routing rule  →  Backend pool
                     ↕                      ↕
              HTTP settings           Health probe
```

| Component | Description |
|---|---|
| **Frontend IP** | Public and/or private IP that clients connect to |
| **Listener** | Listens on a frontend IP:port + protocol; basic (one site) or multi-site (hostname-based) |
| **Request routing rule** | Maps a listener to a backend pool + HTTP settings; basic or path-based |
| **Backend pool** | VMs, VMSS, App Service, IP addresses, or FQDNs |
| **HTTP settings** | Backend protocol, port, cookie-based affinity, connection draining, timeout |
| **Health probe** | Custom HTTP/HTTPS probe to determine backend health |
| **Rewrite rules** | Modify request/response HTTP headers or URL |
| **Redirect rules** | Redirect HTTP → HTTPS or between paths/hostnames |

### Routing types

**Basic routing** — all requests on a listener go to one backend pool:
```
*.contoso.com → backend pool: all-servers
```

**Path-based routing** — route based on URL path:
```
contoso.com/images/*  → backend pool: image-servers
contoso.com/api/*     → backend pool: api-servers
contoso.com/*         → backend pool: web-servers (default)
```

**Multi-site routing** — multiple hostnames on the same Application Gateway:
```
app1.contoso.com  → listener 1 → backend pool 1
app2.contoso.com  → listener 2 → backend pool 2
```

### SSL/TLS termination

| Mode | Description |
|---|---|
| **SSL termination** | Application Gateway decrypts TLS; backend receives plain HTTP |
| **End-to-end SSL** | Application Gateway decrypts, inspects, then re-encrypts to backend |
| **SSL passthrough** | Not supported — App Gateway must decrypt to perform L7 routing |

Certificates are stored in **Azure Key Vault** (recommended) or uploaded directly. App Gateway supports **managed certificates** via Key Vault auto-rotation.

### Web Application Firewall (WAF)

The WAF v2 SKU includes an integrated WAF based on the **OWASP Core Rule Set (CRS)**. It protects against:
- SQL injection
- Cross-site scripting (XSS)
- Command injection
- HTTP protocol violations
- Known CVEs

| Mode | Behaviour |
|---|---|
| **Detection** | Log violations; do not block — use for tuning |
| **Prevention** | Block requests matching rules |

WAF policies can be attached at the Application Gateway level (global) or per listener/routing rule (per-site). You can add custom rules and exclusions to reduce false positives.

### Autoscaling

v2 SKUs scale automatically based on traffic load — from a minimum of 0 instances (scale to zero when idle) up to 125 capacity units. You can set a minimum instance count to pre-warm capacity and avoid cold starts.

## Azure Front Door

**Azure Front Door** is a global L7 load balancer, CDN, and WAF operating at the Microsoft edge (150+ PoPs worldwide). It is the go-to choice when you need to distribute HTTP/HTTPS traffic across multiple Azure regions.

```
Users (global)  →  Front Door PoP  →  Routing  →  Origin (Azure App Service, AKS, IP)
                   (anycast IP)          ↓                  (any region)
                                   WAF policy
```

### Key capabilities

| Feature | Description |
|---|---|
| **Global anycast** | Single IP/FQDN; traffic routed to nearest PoP |
| **Origin groups** | Multiple backends per route with health probes and weighted/latency routing |
| **Caching** | CDN caching at the edge PoP — reduce origin load |
| **WAF** | Global WAF policy applied at the edge |
| **SSL offload** | TLS termination at the PoP |
| **Private Link origins** | Send traffic to private origins without exposing them publicly |
| **Custom domains** | Bring your own domain + managed TLS certificate |

### Routing methods

| Method | Description |
|---|---|
| **Latency** (default) | Route to the origin with the lowest latency |
| **Weighted** | Distribute traffic across origins by weight (canary deployments) |
| **Priority** | Primary origin gets all traffic; fallback to secondary on failure |
| **Session affinity** | Pin a user session to the same origin |

### Front Door vs Application Gateway

| | Application Gateway | Azure Front Door |
|---|---|---|
| Scope | Regional | Global |
| Deployment | In your VNet subnet | Microsoft-managed PoPs |
| Protocols | HTTP/HTTPS | HTTP/HTTPS |
| CDN | No | Yes |
| Private backend access | Yes (VNet) | Yes (Private Link origins) |
| Use case | Single-region web app delivery | Multi-region or global web apps |

## Azure Traffic Manager

**Azure Traffic Manager** is a DNS-based global traffic router. It does not proxy traffic — it returns a DNS response pointing the client to the best endpoint based on a routing policy. The client then connects directly to that endpoint.

```
Client DNS query  →  Traffic Manager  →  DNS response: "use eastus.app.com"
                                          Client connects directly to eastus.app.com
```

### Routing methods

| Method | Description |
|---|---|
| **Performance** | Route to the endpoint with lowest DNS latency from the client |
| **Weighted** | Distribute traffic by weight — for gradual migration or A/B testing |
| **Priority** | All traffic to primary; failover to secondary when primary is unhealthy |
| **Geographic** | Route based on client's geographic region — for data residency compliance |
| **Multivalue** | Return multiple healthy endpoints — client selects |
| **Subnet** | Route based on client IP address range |

### Traffic Manager vs Front Door

| | Traffic Manager | Azure Front Door |
|---|---|---|
| Layer | DNS (not a proxy) | L7 HTTP proxy |
| Protocol support | Any (DNS points to endpoint) | HTTP/HTTPS only |
| SSL termination | No | Yes |
| CDN | No | Yes |
| Failover speed | DNS TTL dependent (60–300 s) | Near-instant (connection-level) |
| Use case | Non-HTTP protocols, DNS-only routing | HTTP/HTTPS global delivery |

## Choosing the Right Load Balancer

```
Is traffic HTTP/HTTPS?
├── No  → Azure Load Balancer (L4)
└── Yes → Is the scope global or multi-region?
           ├── Yes → Azure Front Door (L7 proxy + CDN)
           │         or Traffic Manager (DNS-only, non-HTTP too)
           └── No  → Single region
                      ├── Need WAF / URL routing / SSL termination? → Application Gateway
                      └── Pure L4 / high throughput?               → Azure Load Balancer
```

**Common combinations:**
- **Front Door → Application Gateway** — Front Door handles global routing and edge WAF; App Gateway handles regional URL routing and backend pools inside the VNet
- **Application Gateway → Internal Load Balancer** — App Gateway terminates SSL and routes HTTP to a tier; ILB distributes across app server VMs
- **Traffic Manager → regional endpoints** — DNS failover across regions for non-HTTP workloads

## Working with Load Balancer and Application Gateway via Python

In [ ]:
# pip install azure-identity azure-mgmt-network

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.network.models import (
    LoadBalancer, FrontendIPConfiguration, BackendAddressPool,
    Probe, LoadBalancingRule, SubResource,
    ApplicationGateway, ApplicationGatewayFrontendIPConfiguration,
    ApplicationGatewayFrontendPort, ApplicationGatewayBackendAddressPool,
    ApplicationGatewayBackendHttpSettings, ApplicationGatewayHttpListener,
    ApplicationGatewayRequestRoutingRule, ApplicationGatewaySku,
    ApplicationGatewayIPConfiguration
)

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"
location        = "eastus"

net = NetworkManagementClient(credential, subscription_id)

In [ ]:
# Create a public Standard Load Balancer
pip_id     = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/publicIPAddresses/lb-pip"
frontend   = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/loadBalancers/my-lb/frontendIPConfigurations/frontend"
backend_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/loadBalancers/my-lb/backendAddressPools/backend"
probe_id   = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/loadBalancers/my-lb/probes/http-probe"

lb_params = LoadBalancer(
    location=location,
    sku={"name": "Standard"},
    frontend_ip_configurations=[
        FrontendIPConfiguration(
            name="frontend",
            public_ip_address=SubResource(id=pip_id)
        )
    ],
    backend_address_pools=[
        BackendAddressPool(name="backend")
    ],
    probes=[
        Probe(
            name="http-probe",
            protocol="Http",
            port=80,
            request_path="/health",
            interval_in_seconds=15,
            number_of_probes=2
        )
    ],
    load_balancing_rules=[
        LoadBalancingRule(
            name="http-rule",
            protocol="Tcp",
            frontend_port=80,
            backend_port=80,
            frontend_ip_configuration=SubResource(id=frontend),
            backend_address_pool=SubResource(id=backend_id),
            probe=SubResource(id=probe_id),
            enable_floating_ip=False,
            idle_timeout_in_minutes=4
        )
    ]
)
poller = net.load_balancers.begin_create_or_update(resource_group, "my-lb", lb_params)
lb = poller.result()
print(f"Load balancer created: {lb.name}  SKU: {lb.sku.name}")

In [ ]:
# List load balancers and their rules
for lb in net.load_balancers.list(resource_group):
    print(f"\n{lb.name} ({lb.sku.name})")
    for rule in (lb.load_balancing_rules or []):
        print(f"  rule: {rule.name:<25} {rule.protocol} :{rule.frontend_port} → :{rule.backend_port}")
    for probe in (lb.probes or []):
        print(f"  probe: {probe.name:<24} {probe.protocol} :{probe.port} path: {probe.request_path or '-'}")

In [ ]:
# Create an Application Gateway (WAF v2 SKU) with path-based routing
subnet_id  = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/virtualNetworks/my-vnet/subnets/appgw-subnet"
pip_id     = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/publicIPAddresses/appgw-pip"

# Resource ID shorthand helpers
base = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Network/applicationGateways/my-appgw"

appgw_params = ApplicationGateway(
    location=location,
    sku=ApplicationGatewaySku(name="WAF_v2", tier="WAF_v2", capacity=2),
    gateway_ip_configurations=[
        ApplicationGatewayIPConfiguration(
            name="gwIpConfig",
            subnet=SubResource(id=subnet_id)
        )
    ],
    frontend_ip_configurations=[
        ApplicationGatewayFrontendIPConfiguration(
            name="frontendPublic",
            public_ip_address=SubResource(id=pip_id)
        )
    ],
    frontend_ports=[
        ApplicationGatewayFrontendPort(name="port80", port=80)
    ],
    backend_address_pools=[
        ApplicationGatewayBackendAddressPool(
            name="web-pool",
            backend_addresses=[{"ip_address": "10.0.1.4"}, {"ip_address": "10.0.1.5"}]
        ),
        ApplicationGatewayBackendAddressPool(
            name="api-pool",
            backend_addresses=[{"ip_address": "10.0.2.4"}]
        )
    ],
    backend_http_settings_collection=[
        ApplicationGatewayBackendHttpSettings(
            name="httpSettings",
            port=80,
            protocol="Http",
            cookie_based_affinity="Disabled",
            request_timeout=30
        )
    ],
    http_listeners=[
        ApplicationGatewayHttpListener(
            name="httpListener",
            frontend_ip_configuration=SubResource(id=f"{base}/frontendIPConfigurations/frontendPublic"),
            frontend_port=SubResource(id=f"{base}/frontendPorts/port80"),
            protocol="Http"
        )
    ],
    request_routing_rules=[
        ApplicationGatewayRequestRoutingRule(
            name="routingRule",
            rule_type="Basic",
            priority=100,
            http_listener=SubResource(id=f"{base}/httpListeners/httpListener"),
            backend_address_pool=SubResource(id=f"{base}/backendAddressPools/web-pool"),
            backend_http_settings=SubResource(id=f"{base}/backendHttpSettingsCollection/httpSettings")
        )
    ]
)
poller = net.application_gateways.begin_create_or_update(resource_group, "my-appgw", appgw_params)
print("Application Gateway provisioning started — takes 5-10 minutes")
agw = poller.result()
print(f"App Gateway: {agw.name}  SKU: {agw.sku.name}  state: {agw.provisioning_state}")

In [ ]:
# Check Application Gateway backend health
poller = net.application_gateways.begin_backend_health(resource_group, "my-appgw")
health = poller.result()

for pool in (health.backend_address_pools or []):
    pool_name = pool.backend_address_pool.id.split("/")[-1] if pool.backend_address_pool else "unknown"
    print(f"\nPool: {pool_name}")
    for server in (pool.backend_http_settings_collection or []):
        for s in (server.servers or []):
            print(f"  {s.address:<20} health: {s.health}  reason: {s.health_probe_log or '-'}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Azure Load Balancer | L4 TCP/UDP; 5-tuple hash; Standard SKU required for AZs and outbound rules |
| Frontend IP | Public (internet-facing) or private (internal) |
| Backend pool | VMs or VMSS instances receiving traffic |
| Health probe | HTTP/HTTPS/TCP check; unhealthy instances removed from rotation |
| Inbound NAT rule | Direct port mapping to a specific VM (SSH/RDP to individual instances) |
| Outbound rules | Explicit SNAT configuration for backend pool internet egress |
| HA Ports | Protocol=All, port=0 on internal LB — for NVA load balancing |
| Session persistence | None (5-tuple) / Client IP (2-tuple) / Client IP+protocol (3-tuple) |
| Application Gateway | L7 HTTP/HTTPS; URL path routing, multi-site, SSL termination, WAF |
| App Gateway listener | Basic (one site) or multi-site (hostname-based routing) |
| Path-based routing | Route /api/* and /images/* to different backend pools |
| SSL termination | App Gateway decrypts TLS; backends receive plain HTTP or re-encrypted HTTPS |
| WAF v2 | OWASP CRS; Detection vs Prevention mode; per-site WAF policies |
| App Gateway autoscaling | v2 SKU scales 0–125 capacity units automatically |
| Azure Front Door | Global L7 anycast; CDN; WAF at edge; multi-region origin routing |
| Traffic Manager | DNS-based global routing; works with any protocol; TTL-dependent failover |
| LB decision | L4 = Load Balancer; L7 regional = App Gateway; L7 global = Front Door; DNS = Traffic Manager |